# Proyecto: Cotizador de Derivados FX (USD/MXN), Curva Implícita y Value at Risk (VaR)
**Materia: Administración de Riesgos Financieros**

En este proyecto desarrollamos la solución solicitada para una mesa de divisas y derivados:
1. **Calcular el precio de compra o venta** sobre el tipo de cambio Dólar-Peso (**USD/MXN**), considerando el precio Spot y el diferencial Bid/Ask.
2. Permitir que el cliente elija el **plazo de inversión o cobertura** y el **valor nominal** (en dólares y su equivalente en pesos).
3. **Asignar automáticamente el instrumento derivado según el plazo**:
   - **Corto plazo (2 días o menos)**: Se le asigna una operación de contado **FX Spot** (o Cash $T+0$, Tom $T+1$, Spot estándar $T+2$).
   - **Mayor a 2 días**: Se le asigna un **FX Swap** y se le ofrece como alternativa un **Forward a un mes** (28 días).
   - **Límite de riesgo**: El plazo máximo permitido está acotado a **2 años (730 días)**.
4. **Valuar los derivados usando la Curva Forward en pesos (Curva Implícita en pesos)**:
   - Explicar qué es la curva implícita bajo la paridad de tasas (CIP).
   - Realizar la cotización en **Puntos Forward (Puntos F)**.
5. **Si el cliente acepta el derivado**:
   - Mostrar el resumen del contrato.
   - Proyectar los escenarios de **Pérdidas y Ganancias (P&L)** al vencimiento frente a movimientos del tipo de cambio.
   - Calcular el **Value at Risk (VaR)** de la posición mediante los métodos vistos en clase: **Paramétrico**, **Histórico** y **Montecarlo (12,500 simulaciones)**.


In [1]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configuramos matplotlib para que las graficas se vean limpias y profesionales
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib'
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110

print("Librerías cargadas correctamente: pandas, numpy y matplotlib.")


Librerías cargadas correctamente: pandas, numpy y matplotlib.


## 1. ¿Qué es la Curva Forward en Pesos y la Curva Implícita en Pesos?

### Explicación en palabras sencillas:
Cuando queremos pactar un precio para comprar o vender dólares en una fecha futura (por ejemplo dentro de 30 días o a un año), ese precio **no se calcula tratando de adivinar el futuro ni por especulación**. Se calcula mediante una regla matemática y financiera llamada **Paridad de Tasas de Interés Cubierta (Covered Interest Rate Parity o CIP)**.

La lógica económica es muy clara:
- En México, las tasas de interés libres de riesgo (Cetes o TIIE de Fondeo) pagan alrededor del **10.50% anual**.
- En Estados Unidos, las tasas equivalentes (SOFR o bonos del Tesoro) pagan alrededor del **4.70% anual**.

Si el tipo de cambio futuro fuera igual al de hoy, cualquier inversionista pediría prestado en dólares al 4.70%, cambiaría esos dólares a pesos y los invertiría al 10.50% en México, ganándose la diferencia sin ningún riesgo. Para evitar este arbitraje (dinero gratis sin riesgo), el mercado exige que el dólar futuro cotice más caro que el dólar de hoy. 

La fórmula matemática para calcular el precio Forward teórico $F$ es:

$$F = S \times \frac{1 + r_{MXN} \times \frac{t}{360}}{1 + r_{USD} \times \frac{t}{360}}$$

Donde:
- $S$: Tipo de cambio Spot actual (pesos por dólar).
- $r_{MXN}$: Tasa libre de riesgo en pesos mexicanos para el plazo $t$.
- $r_{USD}$: Tasa libre de riesgo en dólares estadounidenses para el plazo $t$.
- $t$: Número de días del plazo (con convención bancaria Actual/360).

---

### ¿Y qué es la Curva Implícita en Pesos?
En el mercado interbancario real (las mesas de dinero de los bancos y Banxico), muchas veces no se cotiza la tasa directamente, sino que los operadores cotizan en sus pantallas los **Puntos Forward (Puntos F)** o el precio Forward $F$. 

Si nosotros tomamos ese precio Forward de mercado, el precio Spot y la tasa en dólares, podemos **despejar cuál es la tasa de interés en pesos que el mercado cambiario está descontando**:

$$r_{MXN, \text{implícita}} = \left[ \frac{F}{S} \times \left(1 + r_{USD} \times \frac{t}{360}\right) - 1 \right] \times \frac{360}{t}$$

Esta tasa se llama **Tasa Implícita en Pesos**. Representa cuánto cuesta fondearse en pesos "sintéticamente" (pidiendo prestados dólares, cambiándolos a pesos spot y cubriendo la compra futura con un forward).

---

### ¿Qué son los Puntos F (Forward Points)?
Los **Puntos F** son simplemente la diferencia en pesos o centavos entre el precio Forward y el precio Spot:

$$\text{Puntos F} = F - S$$

También se expresan comúnmente en **pips** (multiplicando la diferencia por 10,000):

$$\text{Puntos F (en pips)} = (F - S) \times 10,000$$

Como en México la tasa de interés es más alta que en EE.UU. ($r_{MXN} > r_{USD}$), los puntos forward son **positivos** ($F > S$); es decir, el dólar cotiza con **premio a futuro** frente al peso.


In [2]:
# Cargar la serie histórica de USD/MXN para calcular volatilidad y precios
archivo_historico = 'historico_usdmxn.csv'

if os.path.exists(archivo_historico):
    df_fx = pd.read_csv(archivo_historico)
    df_fx['Date'] = pd.to_datetime(df_fx['Date'])
    spot_actual = float(df_fx['USDMXN'].iloc[-1])
else:
    spot_actual = 20.1000

# Precios de la mesa con spread de compra y venta (spread típico interbancario de 4 centavos)
spread = 0.0400
spot_bid = round(spot_actual - spread / 2, 4)  # Precio al que el banco compra USD (si el cliente vende)
spot_ask = round(spot_actual + spread / 2, 4)  # Precio al que el banco vende USD (si el cliente compra)

print("--- INSUMOS DE MERCADO SPOT USD/MXN ---")
print(f"Tipo de cambio Spot Mid:   ${spot_actual:.4f} MXN/USD")
print(f"Precio Compra Banco (Bid): ${spot_bid:.4f} MXN/USD (Cliente vende USD)")
print(f"Precio Venta Banco (Ask):  ${spot_ask:.4f} MXN/USD (Cliente compra USD)")

# Nodos de las Curvas de Tasas de Mercado (plazo en días vs tasa anualizada)
# México: Curva representativa TIIE Fondeo / Cetes
curva_mxn = {1: 0.1050, 7: 0.1048, 28: 0.1040, 91: 0.1025, 182: 0.1010, 364: 0.0985, 728: 0.0950}
# EE. UU.: Curva SOFR / OIS
curva_usd = {1: 0.0480, 7: 0.0478, 28: 0.0470, 91: 0.0455, 182: 0.0440, 364: 0.0415, 728: 0.0395}

def obtener_tasas(plazo_dias):
    """Interpola linealmente las tasas libres de riesgo para México y EE. UU."""
    t = max(1, min(plazo_dias, 730))
    dias_mx = sorted(curva_mxn.keys())
    tasas_mx = [curva_mxn[d] for d in dias_mx]
    r_mx = float(np.interp(t, dias_mx, tasas_mx))
    
    dias_us = sorted(curva_usd.keys())
    tasas_us = [curva_usd[d] for d in dias_us]
    r_us = float(np.interp(t, dias_us, tasas_us))
    return r_mx, r_us

def calcular_forward_cip(spot, plazo_dias, r_mx=None, r_us=None):
    """Calcula el Forward y los Puntos F con la fórmula de paridad de tasas."""
    if r_mx is None or r_us is None:
        r_mx, r_us = obtener_tasas(plazo_dias)
    factor_mx = 1.0 + r_mx * (plazo_dias / 360.0)
    factor_us = 1.0 + r_us * (plazo_dias / 360.0)
    forward = spot * (factor_mx / factor_us)
    puntos_f = forward - spot
    return forward, puntos_f

def calcular_tasa_implicita(spot, forward, plazo_dias, r_us=None):
    """Despeja la tasa implícita en pesos a partir del precio forward."""
    if plazo_dias <= 0:
        return 0.0
    if r_us is None:
        _, r_us = obtener_tasas(plazo_dias)
    t_factor = plazo_dias / 360.0
    r_impl = ((forward / spot) * (1.0 + r_us * t_factor) - 1.0) / t_factor
    return r_impl


--- INSUMOS DE MERCADO SPOT USD/MXN ---
Tipo de cambio Spot Mid:   $20.1000 MXN/USD
Precio Compra Banco (Bid): $20.0800 MXN/USD (Cliente vende USD)
Precio Venta Banco (Ask):  $20.1200 MXN/USD (Cliente compra USD)


In [3]:
# Construir la tabla de la Curva Forward e Implícita para los plazos estándar del mercado
plazos_estandar = [1, 2, 7, 14, 28, 60, 91, 182, 364, 728]
filas_curva = []

for d in plazos_estandar:
    rmx, rus = obtener_tasas(d)
    fwd, pts = calcular_forward_cip(spot_actual, d, rmx, rus)
    tasa_impl = calcular_tasa_implicita(spot_actual, fwd, d, rus)
    filas_curva.append({
        'Plazo (Días)': d,
        'Tasa MXN (%)': round(rmx * 100, 2),
        'Tasa USD (%)': round(rus * 100, 2),
        'Forward (F)': round(fwd, 4),
        'Puntos F ($)': round(pts, 4),
        'Puntos F (Pips)': round(pts * 10000, 1),
        'Tasa Implícita MXN (%)': round(tasa_impl * 100, 2)
    })

df_curva = pd.DataFrame(filas_curva)
print("--- ESTRUCTURA TEMPORAL: CURVA FORWARD E IMPLÍCITA EN PESOS ---")
try:
    display(df_curva)
except NameError:
    print(df_curva.to_string(index=False))

# Gráfica de la Curva Forward y los Puntos F
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(df_curva['Plazo (Días)'], df_curva['Forward (F)'], marker='o', color='#ff7a1a', linewidth=2.2, label='Tipo de Cambio Forward (F)')
ax1.axhline(spot_actual, color='gray', linestyle='--', label=f'Spot Actual (${spot_actual:.4f})')
ax1.set_title("Curva Forward USD/MXN por Paridad Cubierta de Tasas (CIP)", fontweight='bold')
ax1.set_ylabel("Tipo de Cambio (MXN/USD)", fontweight='bold')
ax1.legend()

ax2.plot(df_curva['Plazo (Días)'], df_curva['Puntos F ($)'], marker='s', color='#2563eb', linewidth=2.0, label='Puntos Forward (F - S en pesos)')
ax2.set_xlabel("Plazo en Días (hasta 2 años)", fontweight='bold')
ax2.set_ylabel("Puntos Forward ($ MXN)", fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()


--- ESTRUCTURA TEMPORAL: CURVA FORWARD E IMPLÍCITA EN PESOS ---
 Plazo (Días)  Tasa MXN (%)  Tasa USD (%)  Forward (F)  Puntos F ($)  Puntos F (Pips)  Tasa Implícita MXN (%)
            1         10.50          4.80      20.1032        0.0032             31.8                   10.50
            2         10.50          4.80      20.1064        0.0064             63.6                   10.50
            7         10.48          4.78      20.1223        0.0223            222.6                   10.48
           14         10.45          4.75      20.1445        0.0445            444.7                   10.45
           28         10.40          4.70      20.1888        0.0888            887.9                   10.40
           60         10.32          4.62      20.2895        0.1895           1894.9                   10.32
           91         10.25          4.55      20.3863        0.2863           2863.1                   10.25
          182         10.10          4.40      20.6666  

## 2. Asignación del Derivado según el Plazo y Cotización Comercial

### ¿Por qué se asigna cada derivado según el plazo?
1. **Corto plazo (2 días o menos)**:
   En los mercados internacionales de divisas, las operaciones de contado (*FX Spot*) liquidan convencionalmente a 48 horas hábiles ($T+2$), en el mismo día (*Cash $T+0$*) o al día siguiente (*Tom $T+1$*). No se necesita contratar una estructura de swap ni un forward para 1 o 2 días porque la liquidación directa de contado cubre esa necesidad de liquidez inmediata.
2. **Plazos mayores a 2 días**:
   Si el cliente requiere una operación a varios días, semanas o meses, se le asigna un **FX Swap**. Este instrumento permite fondear o invertir divisas intercambiándolas hoy a precio spot y pactando simultáneamente la operación inversa a futuro a precio forward, eliminando el riesgo cambiario y aprovechando el diferencial de tasas.
   Además, se le ofrece como alternativa un **Forward a un mes (28 días)** por si solo desea congelar un precio futuro sin mover dinero hoy.
3. **Límite de riesgo a 2 años (730 días)**:
   En el mercado cambiario del peso mexicano, la liquidez y profundidad interbancaria se concentran fuertemente hasta los 2 años. A plazos más largos, el riesgo de crédito y la incertidumbre en tasas aumentan drásticamente, por lo que las políticas de riesgo bancarias limitan estos derivados a un máximo de 730 días.


In [4]:
def cotizar_derivado(tipo_operacion, plazo_dias, nominal_usd=None, nominal_mxn=None):
    """
    Cotiza el precio de compra o venta, valida el plazo y asigna el derivado correspondiente.
    """
    tipo_operacion = tipo_operacion.strip().lower()
    if tipo_operacion not in ['compra', 'venta']:
        raise ValueError("El tipo de operación debe ser 'compra' o 'venta'.")

    # 1. Regla de límite: Plazo acotado a 2 años (730 días)
    alerta_plazo = None
    if plazo_dias > 730:
        alerta_plazo = f"El plazo solicitado ({plazo_dias} días) supera el límite de 2 años (730 días). Se limitó automáticamente a 730 días."
        plazo_dias = 730
    elif plazo_dias < 0:
        raise ValueError("El plazo no puede ser negativo.")

    # 2. Asignación de derivado según el plazo
    if plazo_dias <= 2:
        subtipo = {0: "FX Cash (T+0)", 1: "FX Tom (T+1)", 2: "FX Spot Estándar (T+2)"}.get(plazo_dias, "FX Spot")
        derivado_asignado = f"FX Spot ({subtipo})"
        racional = "Operación de contado directo. Para plazos de 2 días o menos la liquidación estándar de mercado cubre la necesidad sin requerir coberturas a plazo."
        oferta_alt = None
    else:
        derivado_asignado = "FX Swap (Swap Cambiario)"
        racional = f"Al ser un plazo de {plazo_dias} días (> 2 días), el FX Swap permite intercambiar divisas spot y pactar su recompra/reventa forward para fondeo sin riesgo cambiario neto."
        # Ofrecemos complementariamente el Forward a 1 mes (28 días)
        r_mx_1m, r_us_1m = obtener_tasas(28)
        spot_ref = spot_ask if tipo_operacion == 'compra' else spot_bid
        fwd_1m, pts_1m = calcular_forward_cip(spot_ref, 28, r_mx_1m, r_us_1m)
        oferta_alt = {
            'producto': "FX Forward a 1 Mes (28 días)",
            'precio_pactado': round(fwd_1m, 4),
            'puntos_f': round(pts_1m, 4),
            'puntos_f_pips': round(pts_1m * 10000, 1)
        }

    # 3. Determinar precio Spot según el lado de la mesa
    if tipo_operacion == 'compra':
        spot_aplicable = spot_ask  # El cliente compra al precio Venta (Ask) del banco
        lado = "Venta del Banco (Ask)"
    else:
        spot_aplicable = spot_bid  # El cliente vende al precio Compra (Bid) del banco
        lado = "Compra del Banco (Bid)"

    # 4. Cálculo del precio Forward y Puntos F
    if plazo_dias <= 2:
        precio_pactado = spot_aplicable
        puntos_f = 0.0
        tasa_impl = obtener_tasas(plazo_dias)[0]
    else:
        r_mx, r_us = obtener_tasas(plazo_dias)
        fwd, pts = calcular_forward_cip(spot_aplicable, plazo_dias, r_mx, r_us)
        precio_pactado = round(fwd, 4)
        puntos_f = round(pts, 4)
        tasa_impl = calcular_tasa_implicita(spot_aplicable, precio_pactado, plazo_dias, r_us)

    # 5. Cálculo de montos nominales
    if nominal_usd is not None and nominal_usd > 0:
        nom_usd = float(nominal_usd)
        nom_mxn = round(nom_usd * precio_pactado, 2)
    elif nominal_mxn is not None and nominal_mxn > 0:
        nom_mxn = float(nominal_mxn)
        nom_usd = round(nom_mxn / precio_pactado, 2)
    else:
        nom_usd = 100000.0
        nom_mxn = round(nom_usd * precio_pactado, 2)

    return {
        'tipo_operacion': tipo_operacion.upper(),
        'lado': lado,
        'plazo_dias': plazo_dias,
        'alerta_plazo': alerta_plazo,
        'derivado_asignado': derivado_asignado,
        'racional': racional,
        'spot_base': spot_aplicable,
        'puntos_f': puntos_f,
        'puntos_f_pips': round(puntos_f * 10000, 1),
        'precio_pactado': precio_pactado,
        'tasa_implicita_mxn': round(tasa_impl * 100, 2),
        'nominal_usd': nom_usd,
        'nominal_mxn': nom_mxn,
        'oferta_alternativa': oferta_alt
    }


In [5]:
# Ejemplo: Cliente solicita COMPRA de 100,000 USD a un plazo de 30 días
cotizacion = cotizar_derivado(tipo_operacion='compra', plazo_dias=30, nominal_usd=100000)

print("="*68)
print("                   RESUMEN DE COTIZACIÓN COMERCIAL")
print("="*68)
if cotizacion['alerta_plazo']:
    print(f"AVISO: {cotizacion['alerta_plazo']}\n")
print(f"• Operación Solicitada:   {cotizacion['tipo_operacion']} USD ({cotizacion['lado']})")
print(f"• Plazo Elegido:          {cotizacion['plazo_dias']} días")
print(f"• DERIVADO ASIGNADO:      >>> {cotizacion['derivado_asignado']} <<<")
print(f"• Racional de la Mesa:    {cotizacion['racional']}\n")
print(f"• Spot Base Aplicable:    ${cotizacion['spot_base']:.4f} MXN/USD")
print(f"• PUNTOS FORWARD (F):     {cotizacion['puntos_f']:+.4f} MXN  ({cotizacion['puntos_f_pips']:+.1f} pips)")
print(f"• PRECIO PACTADO (F):     ${cotizacion['precio_pactado']:.4f} MXN/USD")
print(f"• Tasa Implícita en MXN:  {cotizacion['tasa_implicita_mxn']:.2f}% anual\n")
print(f"• VALOR NOMINAL EN USD:   ${cotizacion['nominal_usd']:,.2f} USD")
print(f"• VALOR NOMINAL EN MXN:   ${cotizacion['nominal_mxn']:,.2f} MXN")

if cotizacion['oferta_alternativa']:
    alt = cotizacion['oferta_alternativa']
    print("-"*68)
    print(f"• Oferta Alternativa:     {alt['producto']}")
    print(f"• Precio Forward 1 Mes:   ${alt['precio_pactado']:.4f} MXN (Puntos F: {alt['puntos_f']:+.4f})")
print("="*68)


                   RESUMEN DE COTIZACIÓN COMERCIAL
• Operación Solicitada:   COMPRA USD (Venta del Banco (Ask))
• Plazo Elegido:          30 días
• DERIVADO ASIGNADO:      >>> FX Swap (Swap Cambiario) <<<
• Racional de la Mesa:    Al ser un plazo de 30 días (> 2 días), el FX Swap permite intercambiar divisas spot y pactar su recompra/reventa forward para fondeo sin riesgo cambiario neto.

• Spot Base Aplicable:    $20.1200 MXN/USD
• PUNTOS FORWARD (F):     +0.0952 MXN  (+952.0 pips)
• PRECIO PACTADO (F):     $20.2152 MXN/USD
• Tasa Implícita en MXN:  10.40% anual

• VALOR NOMINAL EN USD:   $100,000.00 USD
• VALOR NOMINAL EN MXN:   $2,021,520.00 MXN
--------------------------------------------------------------------
• Oferta Alternativa:     FX Forward a 1 Mes (28 días)
• Precio Forward 1 Mes:   $20.2089 MXN (Puntos F: +0.0889)


## 3. Si el cliente acepta el derivado: Proyecciones de Pérdidas y Ganancias (P&L)

Cuando el cliente decide contratar el derivado al precio pactado $F$:
- Si el cliente **compró dólares**, su ganancia o pérdida al vencimiento dependerá de dónde termine el tipo de cambio de mercado ($S_T$). Si el dólar sube ($S_T > F$), gana dinero porque compró más barato que el mercado; si baja ($S_T < F$), tiene pérdida:
  $$\text{P\&L (Compra USD)} = \text{Nominal USD} \times (S_T - F)$$
- Si el cliente **vendió dólares**, gana si el dólar baja / el peso se aprecia ($S_T < F$):
  $$\text{P\&L (Venta USD)} = \text{Nominal USD} \times (F - S_T)$$

A continuación proyectamos los resultados de pérdidas y ganancias para variaciones desde $-15\%$ hasta $+15\%$ del tipo de cambio al vencimiento.


In [6]:
# Matriz de Escenarios de P&L al vencimiento
variaciones = [-0.15, -0.10, -0.05, -0.02, 0.0, 0.02, 0.05, 0.10, 0.15]
F = cotizacion['precio_pactado']
nom_usd = cotizacion['nominal_usd']
nom_mxn = cotizacion['nominal_mxn']
es_compra = (cotizacion['tipo_operacion'] == 'COMPRA')

filas_pnl = []
for v in variaciones:
    st = round(F * (1.0 + v), 4)
    if es_compra:
        pnl_mxn = round(nom_usd * (st - F), 2)
    else:
        pnl_mxn = round(nom_usd * (F - st), 2)
    
    pnl_usd = round(pnl_mxn / st, 2)
    rend_pct = round((pnl_mxn / nom_mxn) * 100, 2)
    estado = "GANANCIA" if pnl_mxn > 0 else ("PÉRDIDA" if pnl_mxn < 0 else "PRECIO PACTADO")
    
    filas_pnl.append({
        'Variación (%)': f"{v*100:+.1f}%",
        'ST Proyectado': st,
        'Flujo Pactado (MXN)': nom_mxn,
        'Flujo Mercado (MXN)': round(nom_usd * st, 2),
        'P&L (MXN)': pnl_mxn,
        'P&L (USD)': pnl_usd,
        'Rendimiento (%)': rend_pct,
        'Estado': estado
    })

df_pnl = pd.DataFrame(filas_pnl)
print("--- PROYECCIÓN DE ESCENARIOS DE P&L AL VENCIMIENTO ---")
try:
    display(df_pnl)
except NameError:
    print(df_pnl.to_string(index=False))

# Gráfica del perfil de Payoff de Pérdidas y Ganancias
plt.figure(figsize=(9, 4.5))
plt.plot(df_pnl['ST Proyectado'], df_pnl['P&L (MXN)'], marker='o', color='#ff7a1a', linewidth=2.5, label='Perfil de P&L')
plt.axhline(0, color='black', linestyle='--', linewidth=1.0)
plt.axvline(F, color='red', linestyle=':', linewidth=1.8, label=f'Precio Pactado F: ${F:.4f}')

# Sombrear zonas de ganancia y pérdida
plt.fill_between(df_pnl['ST Proyectado'], df_pnl['P&L (MXN)'], 0, where=(df_pnl['P&L (MXN)'] >= 0), color='#10b981', alpha=0.25, label='Zona de Ganancia')
plt.fill_between(df_pnl['ST Proyectado'], df_pnl['P&L (MXN)'], 0, where=(df_pnl['P&L (MXN)'] < 0), color='#f43f5e', alpha=0.25, label='Zona de Pérdida')

plt.title(f"Perfil de Payoff P&L - Posición {cotizacion['tipo_operacion']} USD ({cotizacion['derivado_asignado']})", fontweight='bold')
plt.xlabel("Tipo de Cambio al Vencimiento ST (MXN/USD)", fontweight='bold')
plt.ylabel("P&L en Pesos (MXN)", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


--- PROYECCIÓN DE ESCENARIOS DE P&L AL VENCIMIENTO ---
Variación (%)  ST Proyectado  Flujo Pactado (MXN)  Flujo Mercado (MXN)  P&L (MXN)  P&L (USD)  Rendimiento (%)         Estado
       -15.0%        17.1829            2021520.0            1718290.0  -303230.0  -17647.20            -15.0        PÉRDIDA
       -10.0%        18.1937            2021520.0            1819370.0  -202150.0  -11110.99            -10.0        PÉRDIDA
        -5.0%        19.2044            2021520.0            1920440.0  -101080.0   -5263.38             -5.0        PÉRDIDA
        -2.0%        19.8109            2021520.0            1981090.0   -40430.0   -2040.80             -2.0        PÉRDIDA
        +0.0%        20.2152            2021520.0            2021520.0        0.0       0.00              0.0 PRECIO PACTADO
        +2.0%        20.6195            2021520.0            2061950.0    40430.0    1960.77              2.0       GANANCIA
        +5.0%        21.2260            2021520.0            2122600.0

## 4. Cálculo del Value at Risk (VaR) sobre la Posición Contratada

### ¿Qué representa el VaR para el cliente?
El **Value at Risk (VaR)** responde a la pregunta de gestión de riesgos: *¿Cuánto es lo máximo que puedo esperar perder en condiciones normales de mercado durante un periodo de tiempo determinado y bajo un nivel de confianza estadístico conocido?*

Aplicamos exactamente las 3 metodologías vistas en los ejercicios anteriores:
1. **VaR Paramétrico (Delta-Normal y Escalado a 10 días)**:
   Calculamos la desviación estándar diaria $\sigma_{diaria}$ del tipo de cambio. Para un horizonte de 10 días (estándar bancario regulatorio), escalamos con la regla de la raíz del tiempo:
   $$\sigma_{10d} = \sigma_{diaria} \times \sqrt{10}$$
   Usamos los valores críticos de la distribución normal: $Z = 1.645$ para 95% de confianza y $Z = 2.326$ para 99% de confianza:
   $$\text{VaR Paramétrico} = Z \times \sigma_{10d} \times \text{Nominal}$$

2. **VaR Histórico (1 día)**:
   Tomamos la serie real histórica diaria de rendimientos de USD/MXN y encontramos el percentil 5% (para 95% de confianza) o percentil 1% (para 99% de confianza). No asume ninguna distribución teórica, sino lo que realmente ocurrió en el mercado.

3. **VaR Montecarlo (10 días con 12,500 simulaciones)**:
   Siguiendo el mismo estándar que usamos en los exámenes y ejercicios de la carpeta, generamos **$N = 12,500$ escenarios estocásticos** de rendimientos a 10 días para evaluar la distribución de pérdidas simuladas y obtener los percentiles de cola al 95% y 99%.


In [7]:
# 1. Volatilidad histórica diaria a partir de la serie de USD/MXN
retornos = df_fx['USDMXN'].pct_change().dropna()
sigma_diaria = float(retornos.std())
mu_diaria = float(retornos.mean())
plazo_d = cotizacion['plazo_dias']

sigma_10d = sigma_diaria * math.sqrt(10)
z_95 = 1.64485
z_99 = 2.32635

# A) VaR Paramétrico a 10 días
var_p95_10d = z_95 * sigma_10d * nom_mxn
var_p99_10d = z_99 * sigma_10d * nom_mxn

# B) VaR Histórico a 1 día
ret_posicion = retornos if es_compra else -retornos
var_h95_1d = abs(float(np.percentile(ret_posicion, 5))) * nom_mxn
var_h99_1d = abs(float(np.percentile(ret_posicion, 1))) * nom_mxn

# C) VaR Montecarlo a 10 días (12,500 simulaciones)
np.random.seed(42)
n_simulaciones = 12500
shocks = np.random.normal(0, 1, n_simulaciones)
sim_ret_10d = mu_diaria * 10 + sigma_10d * shocks
sim_pnl_pct = sim_ret_10d if es_compra else -sim_ret_10d
sim_pnl_mxn = sim_pnl_pct * nom_mxn

var_mc95_10d = abs(float(np.percentile(sim_pnl_mxn, 5)))
var_mc99_10d = abs(float(np.percentile(sim_pnl_mxn, 1)))

print("="*68)
print("                   ANÁLISIS DE VALUE AT RISK (VaR)")
print("="*68)
print(f"• Volatilidad Diaria del USD/MXN:     {sigma_diaria*100:.2f}% (Anualizada: {sigma_diaria*math.sqrt(252)*100:.2f}%)\n")

print("1. VaR Paramétrico (Horizonte 10 Días):")
print(f"   - Confianza 95%: ${var_p95_10d:,.2f} MXN  ({(var_p95_10d/nom_mxn)*100:.2f}%)")
print(f"   - Confianza 99%: ${var_p99_10d:,.2f} MXN  ({(var_p99_10d/nom_mxn)*100:.2f}%)\n")

print("2. VaR Histórico (Horizonte 1 Día):")
print(f"   - Confianza 95%: ${var_h95_1d:,.2f} MXN  ({(var_h95_1d/nom_mxn)*100:.2f}%)")
print(f"   - Confianza 99%: ${var_h99_1d:,.2f} MXN  ({(var_h99_1d/nom_mxn)*100:.2f}%)\n")

print("3. VaR Montecarlo (10 Días con 12,500 Simulaciones):")
print(f"   - Confianza 95%: ${var_mc95_10d:,.2f} MXN  ({(var_mc95_10d/nom_mxn)*100:.2f}%)")
print(f"   - Confianza 99%: ${var_mc99_10d:,.2f} MXN  ({(var_mc99_10d/nom_mxn)*100:.2f}%)")
print("="*68)

# Histograma de las 12,500 simulaciones Montecarlo con los cortes de VaR
plt.figure(figsize=(9, 4.5))
plt.hist(sim_pnl_mxn, bins=80, density=True, color='#2563eb', alpha=0.7, edgecolor='#1e40af', label='Simulaciones Montecarlo (N=12,500)')
plt.axvline(-var_mc95_10d, color='#ff7a1a', linestyle='--', linewidth=2.0, label=f'VaR 95% 10d: -${var_mc95_10d:,.0f} MXN')
plt.axvline(-var_mc99_10d, color='#ef4444', linestyle='-', linewidth=2.2, label=f'VaR 99% 10d: -${var_mc99_10d:,.0f} MXN')

plt.title("Distribución de Pérdidas y Ganancias Simuladas y Cortes de VaR (10 Días)", fontweight='bold')
plt.xlabel("Pérdida / Ganancia Simulada en Pesos (MXN)", fontweight='bold')
plt.ylabel("Densidad", fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


                   ANÁLISIS DE VALUE AT RISK (VaR)
• Volatilidad Diaria del USD/MXN:     0.68% (Anualizada: 10.84%)

1. VaR Paramétrico (Horizonte 10 Días):
   - Confianza 95%: $71,823.27 MXN  (3.55%)
   - Confianza 99%: $101,581.33 MXN  (5.02%)

2. VaR Histórico (Horizonte 1 Día):
   - Confianza 95%: $20,842.50 MXN  (1.03%)
   - Confianza 99%: $28,089.71 MXN  (1.39%)

3. VaR Montecarlo (10 Días con 12,500 Simulaciones):
   - Confianza 95%: $66,961.13 MXN  (3.31%)
   - Confianza 99%: $98,310.77 MXN  (4.86%)


## 5. Conclusiones y Resumen para el Cliente

1. **Sobre el Derivado Asignado**:
   Al elegir un plazo de 30 días, el sistema asignó automáticamente un **FX Swap** por ser la estructura óptima de mercado para fondearse e intercambiar divisas eliminando el riesgo cambiario neto. Además, se le presentó la opción de un **Forward a 1 mes** a un precio de $20.2089 MXN/USD si solo busca fijar el tipo de cambio sin flujos inmediatos.
2. **Sobre los Puntos F y la Curva Implícita**:
   El precio forward pactado de **$20.2152 MXN/USD** incorpora **+0.0952 puntos forward (+952 pips)** sobre el spot. Este premio refleja fielmente el diferencial entre las tasas de interés en México (10.40%) y EE. UU. (4.70%) según la paridad cubierta de tasas (CIP), garantizando una cotización de mercado justa y sin arbitraje.
3. **Sobre el Riesgo y el VaR**:
   Con un nivel de confianza del **99% a 10 días**, el modelo de Montecarlo (12,500 simulaciones) proyecta que la pérdida máxima esperada bajo condiciones normales de mercado no superará los **$98,310.77 MXN** (aproximadamente un 4.86% del valor nominal contratado). Este dato le permite a la tesorería definir con precisión sus reservas de liquidez y garantías colaterales.
